# 04.9 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/09-bias-variance/exercises.ipynb)

## Exercise 1: by hand

True value 50. Model C predicts 60, 62, 58, 60 across four retrains. Model D predicts 40, 65, 45, 50. Compute bias, bias², variance and MSE for each. Which would you choose?

In [ ]:
# your code here

## Exercise 2: more noise

Change the simulation's noise from 20 to 60 and recompute bias² and variance for degrees 1, 3 and 9. Which degree is best now? Why does extra noise favour simpler models?

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)
def true_f(x):
    """The real (unknown) relationship: spend rises with visits, then levels off."""
    return 200 * (1 - np.exp(-x / 4))
x_grid = np.linspace(1.5, 13, 60)          # where we compare models (inside the data range)

def one_sample(n=30, noise=20):
    x = rng.uniform(0.5, 14, n)
    return x, true_f(x) + rng.normal(0, noise, n)

In [ ]:
# your code here

## Exercise 3: `min_samples_leaf`

On the real churn training set, draw a validation curve for `DecisionTreeClassifier(min_samples_leaf=…)` over [1, 2, 5, 10, 25, 50, 100, 200, 400]. Which end of the curve is overfitting?

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

In [ ]:
# your code here

## Exercise 4: a learning curve for a depth-5 tree

Plot the learning curve for `DecisionTreeClassifier(max_depth=5)`. Compare its gap with the unlimited tree's.

In [ ]:
# your code here